#### 读取包含缺失值的Iris数据集

In [72]:
from matplotlib.ticker import scale_range
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [73]:
spark = SparkSession.builder.appName("IrisMissingValue").getOrCreate()

In [74]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("file:///D:/code/python/BigDataDevelopment/myCode/iris3.csv")

#### 识别包含缺失值的记录

In [75]:
df_missing = df.filter(col("sepal_length").isNull() | col("sepal_width").isNull() | col("petal_length").isNull())
df_missing.show()

+------------+-----------+------------+-----------+----------+
|sepal_length|sepal_width|petal_length|petal_width|   species|
+------------+-----------+------------+-----------+----------+
|        null|        3.0|         1.4|        0.1|    setosa|
|         6.5|       null|         4.6|        1.5|versicolor|
|         6.2|        2.8|        null|        1.8| virginica|
+------------+-----------+------------+-----------+----------+



#### 均值填充

In [76]:
mean_fill = df.groupBy("species").agg(
    round(mean("sepal_length"), 1).alias("mean_sl"),
    round(mean("sepal_width"), 1).alias("mean_sw"),
    round(mean("petal_length"), 1).alias("mean_pl")
)
mean_fill.show()

+----------+-------+-------+-------+
|   species|mean_sl|mean_sw|mean_pl|
+----------+-------+-------+-------+
| virginica|    6.6|    3.0|    5.6|
|versicolor|    5.9|    2.8|    4.3|
|    setosa|    5.0|    3.4|    1.5|
+----------+-------+-------+-------+



In [77]:
df_filled_mean = df.join(mean_fill, "species", "left").select(
    coalesce(col("sepal_length"), col("mean_sl")).alias("sepal_length"),
    coalesce(col("sepal_width"), col("mean_sw")).alias("sepal_width"),
    coalesce(col("petal_length"), col("mean_pl")).alias("petal_length"),
    col("petal_width"),
    col("species")
)

In [78]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy(monotonically_increasing_id())
df_with_index = df_filled_mean.withColumn("row_num", row_number().over(window))

print("按行号检验 (第14, 56, 128行):")
df_with_index.filter(col("row_num").isin(13, 55, 127)).show()

按行号检验 (第14, 56, 128行):
+------------+-----------+------------+-----------+----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|   species|row_num|
+------------+-----------+------------+-----------+----------+-------+
|         5.0|        3.0|         1.4|        0.1|    setosa|     13|
|         6.5|        2.8|         4.6|        1.5|versicolor|     55|
|         6.2|        2.8|         5.6|        1.8| virginica|    127|
+------------+-----------+------------+-----------+----------+-------+



#### KNN填充

In [79]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("file:///D:/code/python/BigDataDevelopment/myCode/iris3.csv")

In [85]:
missing_rows = df.filter(col("sepal_length").isNull() | col("sepal_width").isNull() | col("petal_length").isNull())

In [81]:
complete_rows = df.filter(~col("sepal_length").isNull() & ~col("sepal_width").isNull() & ~col("petal_length").isNull())

In [82]:
knn_fill_values = {}
for missing_row in missing_rows.collect():
    row_data = missing_row.asDict()
    species = row_data["species"]

    missing_col = None
    if row_data["sepal_length"] is None:
        missing_col = "sepal_length"
    elif row_data["sepal_width"] is None:
        missing_col = "sepal_width"
    elif row_data["petal_length"] is None:
        missing_col = "petal_length"

    same_species = complete_rows.filter(col("species") == species)

    distance_expr = None
    if missing_col == "sepal_length":
        distance_expr = sqrt(
            pow(col("sepal_width") - row_data["sepal_width"], 2) +
            pow(col("petal_length") - row_data["petal_length"], 2) +
            pow(col("petal_width") - row_data["petal_width"], 2)
        )
    elif missing_col == "sepal_width":
        distance_expr = sqrt(
            pow(col("sepal_length") - row_data["sepal_length"], 2) +
            pow(col("petal_length") - row_data["petal_length"], 2) +
            pow(col("petal_width") - row_data["petal_width"], 2)
        )
    elif missing_col == "petal_length":
        distance_expr = sqrt(
            pow(col("sepal_length") - row_data["sepal_length"], 2) +
            pow(col("sepal_width") - row_data["sepal_width"], 2) +
            pow(col("petal_width") - row_data["petal_width"], 2)
        )

    nearest = same_species.withColumn("distance", distance_expr).orderBy("distance").first()

    knn_fill_values[(species, missing_col)] = nearest[missing_col]
    print(f"类别 {species} 的缺失列 {missing_col} 填充值: {nearest[missing_col]}")

类别 setosa 的缺失列 sepal_length 填充值: 4.9
类别 versicolor 的缺失列 sepal_width 填充值: 3.2
类别 virginica 的缺失列 petal_length 填充值: 4.9


In [83]:
df_knn_filled = df
for (species, missing_col), fill_value in knn_fill_values.items():
    df_knn_filled = df_knn_filled.withColumn(
        missing_col,
        when((col("species") == species) & (col(missing_col).isNull()), fill_value).otherwise(col(missing_col))
    )

In [84]:
window = Window.orderBy(monotonically_increasing_id())
df_knn_with_index = df_knn_filled.withColumn("row_num", row_number().over(window))
print("KNN填充后的缺失行检验 (第14, 56, 128行):")
df_knn_with_index.filter(col("row_num").isin(13, 55, 127)).show()

KNN填充后的缺失行检验 (第14, 56, 128行):
+------------+-----------+------------+-----------+----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|   species|row_num|
+------------+-----------+------------+-----------+----------+-------+
|         4.9|        3.0|         1.4|        0.1|    setosa|     13|
|         6.5|        3.2|         4.6|        1.5|versicolor|     55|
|         6.2|        2.8|         4.9|        1.8| virginica|    127|
+------------+-----------+------------+-----------+----------+-------+

